In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import pandas as pd
import os

In [3]:
os.getcwd()

'C:\\Users\\rajan\\PycharmProjects\\Book-Recommendor'

In [4]:
load_dotenv()

True

In [21]:
books = pd.read_csv('books_dataset_cleaned.csv')
books.drop(["subtitle", "description_words_count"], inplace=True, axis=1)

## Text Splitting

In [6]:
books["tagged_description"].to_csv("tagged_description.txt", sep="\n", header=False, index=False)

In [7]:
# Cleaning text descriptions
with open("tagged_description.txt", encoding='utf-8') as f:
    lines = f.readlines()

cleaned_lines = [line.strip().strip('"') for line in lines]
cleaned_text = "\n".join(cleaned_lines)
with open("cleaned_description.txt", "w", encoding='utf-8') as f:
    f.write(cleaned_text)

In [8]:
raw_documents = TextLoader("cleaned_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 0
Created a chunk of size 1214, which is longer than the specified 0
Created a chunk of size 373, which is longer than the specified 0
Created a chunk of size 309, which is longer than the specified 0
Created a chunk of size 481, which is longer than the specified 0
Created a chunk of size 482, which is longer than the specified 0
Created a chunk of size 960, which is longer than the specified 0
Created a chunk of size 188, which is longer than the specified 0
Created a chunk of size 843, which is longer than the specified 0
Created a chunk of size 294, which is longer than the specified 0
Created a chunk of size 195, which is longer than the specified 0
Created a chunk of size 879, which is longer than the specified 0
Created a chunk of size 1088, which is longer than the specified 0
Created a chunk of size 1189, which is longer than the specified 0
Created a chunk of size 304, which is longer than the specified 0
Create

In [9]:
documents[0].page_content

'9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the wors

## Embeddings

In [10]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
db_books = Chroma.from_documents(documents, embedding=embeddings)

In [14]:
query = "A book about Roman Empire"
docs = db_books.similarity_search(query, k=5)
docs

[Document(id='541894f0-e490-4cee-b3a2-a6f4962a2500', metadata={'source': 'cleaned_description.txt'}, page_content='9780688093686 A story tracing the creation of Republican Rome presents those who founded an empire, including Marius and Sulla, each determined to become the First Man of Rome'),
 Document(id='31dab48e-a9a7-4228-b57c-b0e40a8f93bf', metadata={'source': 'cleaned_description.txt'}, page_content='9780679724773 The emperor Claudius tells of his life during the reigns of Augustus, Tiberius, and Caligula and the events that led to his rise to power in a classic novel reconstructing ancient Rome'),
 Document(id='c3601de7-a992-445e-bc5f-b373e9f344a6', metadata={'source': 'cleaned_description.txt'}, page_content='9780671024208 Now in paperback--the sweeping epic of ancient Rome from the #1 bestselling author of ""The Thorn Birds"" that brings to life Gaius Julius Caesar during the last days of the Roman Republic.'),
 Document(id='11978a71-9cd9-4b5d-895c-7a7aea379116', metadata={'sou

In [15]:
books[books["isbn13"] == int(docs[0].page_content.split()[0])]

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,description_words_count,title_and_subtitle,tagged_description
3182,9780688093686,068809368X,The first man in Rome,NaN,Colleen McCullough,Fiction,http://books.google.com/books/content?id=kMRmP...,A story tracing the creation of Republican Rom...,1990.0,4.1,896.0,294.0,27,The first man in Rome,9780688093686 A story tracing the creation of ...


In [18]:
def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)

    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.split()[0])]

    return books[books["isbn13"].isin(books_list)].head(top_k)

In [22]:
retrieve_semantic_recommendations("Books about Roman Empire")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
115,9780060510855,0060510854,Caesar,Colleen McCullough,Fiction,http://books.google.com/books/content?id=aoH_-...,A fictional portrait of Julius Caesar follows ...,2003.0,4.36,928.0,5825.0,Caesar: A Novel,9780060510855 A fictional portrait of Julius C...
671,9780140442410,0140442413,Germania,Cornelius Tacitus,History,http://books.google.com/books/content?id=-TCOs...,The Agricola is both a portrait of Julius Agri...,1970.0,3.98,174.0,3895.0,Germania,9780140442410 The Agricola is both a portrait ...
698,9780140449082,0140449086,The Histories,Herodotus,History,http://books.google.com/books/content?id=x-gAo...,One of the masterpieces of classical literatur...,2003.0,3.98,716.0,32923.0,The Histories,9780140449082 One of the masterpieces of class...
887,9780156001267,0156001268,The Metamorphoses of Ovid,Allen Mandelbaum,Literary Criticism,http://books.google.com/books/content?id=HK9gj...,A new translation of the most famous work of a...,1995.0,4.05,559.0,685.0,The Metamorphoses of Ovid,9780156001267 A new translation of the most fa...
905,9780156029063,0156029065,Baudolino,Umberto Eco,Fiction,http://books.google.com/books/content?id=VSLOZ...,"Born a simple peasant in northern Italy, Baudo...",2003.0,3.74,527.0,15490.0,Baudolino,9780156029063 Born a simple peasant in norther...
957,9780192824257,0192824252,The Histories,Herodotus,History,http://books.google.com/books/content?id=VrV5T...,Provides a new translation of the classic acco...,1998.0,3.98,772.0,265.0,The Histories,9780192824257 Provides a new translation of th...
963,9780192833006,0192833006,Agricola and Germany,Tacitus,Literary Collections,http://books.google.com/books/content?id=wS7IJ...,"Cornelius Tacitus, Rome's greatest historian, ...",1999.0,3.98,224.0,25.0,Agricola and Germany,"9780192833006 Cornelius Tacitus, Rome's greate..."
978,9780192837684,0192837680,The Eclogues ; The Georgics,Virgil,Agriculture,http://books.google.com/books/content?id=cbVpH...,"The Eclogues, ten short pastoral poems, were c...",1999.0,3.82,180.0,237.0,The Eclogues ; The Georgics,"9780192837684 The Eclogues, ten short pastoral..."
984,9780192839572,0192839578,La chartreuse de Parme,Jm Stendhal,Fiction,http://books.google.com/books/content?id=9tysD...,Follows the adventures of young Fabrizio del D...,1999.0,3.82,560.0,51.0,La chartreuse de Parme: Apprendre l' anglais e...,9780192839572 Follows the adventures of young ...
1002,9780195135923,019513592X,The Oresteia,Aeschylus,Drama,http://books.google.com/books/content?id=SY9Hl...,The story of the house of Atreus is a tale of ...,2003.0,4.01,304.0,76.0,The Oresteia,9780195135923 The story of the house of Atreus...
